<a href="https://colab.research.google.com/github/dokunoale/chagas/blob/CNN-model/CNN-model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone --branch CNN-model --single-branch https://github.com/dokunoale/chagas.git
%cd chagas
# Carica librerie installate
!pip install wfdb -q
!pip install gdown -q

# Aggiungi solo la root del progetto (src)
import sys
sys.path.append('/content/chagas/src')

# Importa tutto dai moduli
from preprocessing import tf_dataset_loader
from models import libraries, functions, split_dataset
from models.CNN import build_model

# Importa simboli specifici (se vuoi)
from models.libraries import *
from models.functions import *
from models.split_dataset import *
from models.CNN.build_model import *


Cloning into 'chagas'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 110 (delta 31), reused 40 (delta 20), pack-reused 45 (from 1)
Receiving objects: 100% (110/110), 29.15 KiB | 445.00 KiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/chagas
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 60.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.0 which is incompatible.
dask-cudf-cu12 25.2.2 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.0 which is incompatible.
cudf-cu12 25.2.1 requires pandas<

In [2]:
from google.colab import drive
drive.mount('/content/drive')

url = "https://drive.google.com/uc?id=1P3blYg3gBMluOazE69ivQFyzHs7tSeq_"
gdown.download(url, "dataset.zip", quiet=False, fuzzy=True)
!unzip -q dataset.zip -d ./dataset

Mounted at /content/drive


Downloading...
From (original): https://drive.google.com/uc?id=1P3blYg3gBMluOazE69ivQFyzHs7tSeq_
From (redirected): https://drive.google.com/uc?id=1P3blYg3gBMluOazE69ivQFyzHs7tSeq_&confirm=t&uuid=5b6a12fa-3beb-4151-a884-e2fb411ffe2e
To: /content/chagas/dataset.zip
100%|██████████| 435M/435M [00:05<00:00, 73.6MB/s]


In [3]:
X_pos, y_pos = tf_dataset_loader.load_dataset('/content/chagas/dataset/preprocessed/positives')
X_neg, y_neg = tf_dataset_loader.load_dataset('/content/chagas/dataset/preprocessed/negatives')

In [4]:
#Uniamo i positivi e i negativi
X = np.concatenate([X_pos, X_neg], axis=0)
y = np.concatenate([y_pos, y_neg], axis=0)

# Facciamo lo shuffle
indices = np.arange(X.shape[0])
np.random.shuffle(indices)

# Riordinamento di X e y con gli stessi indici
X = X[indices]
y = y[indices]

In [5]:
#SPLIT DEL DATASET
X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(X, y)

In [14]:
# mettiamo le etichette sottoforma di 0 e 1
y_train = y_train.astype(int)
y_test = y_test.astype(int)

print(np.unique(y_train))

[0 1]


# Primo modello: usiamo la BCE

In [13]:
model1 = build_cnn_ecg_model()

#compiliamo il modello
model1.compile(optimizer='adam',
              loss=BinaryCrossentropy(),
              metrics=['accuracy', AUC(name='auc')])

#addestriamo il modello
callback = make_callback("1_CNN")

cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = {0: cw[0], 1: cw[1]}

history1 = model1.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=30,
                    batch_size=32,
                    class_weight=class_weights,
                    callbacks=callback)



In [ ]:
#visualizziamo l'andamento delle metriche durante l'addestramento
plot_training_metrics(history1)

In [ ]:
#Facciamo le predizioni
y_pred_proba = model1.predict(X_test).flatten()

#Troviamo la soglia ottimale
optimal_threshold = find_optimal_threshold(y_true, y_pred_proba)
print(f"Soglia ottimale: {optimal_threshold}")

# Applica soglia ottimale per binarizzare le predizioni
y_pred_binary = (y_pred_proba >= optimal_threshold).astype(int)

In [ ]:
#Calcoliamo la matrice di confusione e la visualizziamo
cm1 = confusion_matrix(y_true, y_pred_binary)
show_confusion_matrix(cm1)

In [ ]:
#Vediamo i risultati
acc1 = accuracy_score(y_test, y_pred_binary)
auc1 = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy: {acc1:.3f}")
print(f"AUC: {auc1:.3f}")

class_report1 = classification_report(y_test, y_pred_binary, target_names=["Negativo", "Positivo"])
print(class_report1)

# Secondo modello: usiamo la focal loss

In [ ]:
model2 = build_cnn_ecg_model()

#compiliamo il modello
model2.compile(optimizer='adam',
              loss=BinaryCrossentropy(),
              metrics=['accuracy', AUC(name='auc')])

#addestriamo il modello
callback = make_callback("2_CNN")

cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = {0: cw[0], 1: cw[1]}

history2 = model2.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=30,
                    batch_size=32,
                    class_weight=class_weights,
                    callbacks=callback)

In [ ]:
#visualizziamo l'andamento delle metriche durante l'addestramento
plot_training_metrics(history2)

In [ ]:
#Facciamo le predizioni
y_pred_proba = model2.predict(X_test).flatten()

#Troviamo la soglia ottimale
optimal_threshold = find_optimal_threshold(y_true, y_pred_proba)
print(f"Soglia ottimale: {optimal_threshold}")

# Applica soglia ottimale per binarizzare le predizioni
y_pred_binary = (y_pred_proba >= optimal_threshold).astype(int)

In [ ]:
#Calcoliamo la matrice di confusione e la visualizziamo
cm2 = confusion_matrix(y_true, y_pred_binary)
show_confusion_matrix(cm2)

In [ ]:
#Vediamo i risultati
acc2 = accuracy_score(y_test, y_pred_binary)
auc2 = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy: {acc2:.3f}")
print(f"AUC: {auc2:.3f}")

class_report2 = classification_report(y_test, y_pred_binary, target_names=["Negativo", "Positivo"])
print(class_report2)